# Step 4 — Market-Driven Cost Optimization

PuLP chooses when each depot charges using SE1 prices while preserving the same fleet, daily-energy, charger and Step-3 network-feasibility constraints. Pandapower then validates the optimized schedule physically.

### What this cell does — Load optimization inputs

Loads the fleet, SE1 prices and `network_limits.csv` created by Step 3. The loader deliberately refuses to continue if Step 3 found the installed charger design infeasible.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config
from src.data import load_fleet, load_prices, load_network_limits, load_base_loads, load_bus_mapping
from src.optimization.model import optimize_cost
from src.network.model import build_network
from src.network.timeseries import run_timeseries, build_snapshot_network
from src.network.contingency import run_n1, SWITCH_CONFIGS_ALL_CLOSED
from src.reporting.metrics import scenario_metrics
from src.reporting.plots import save_schedule_plot, save_network_plot

fleet = load_fleet()
prices = load_prices()
network_limits = load_network_limits()
base_p, base_q = load_base_loads()
bus_map = load_bus_mapping(require_shared=True)
network_factory = lambda: build_network(halve_existing_loads=True, close_ring_switches=True)


### What this cell does — Solve PuLP cost optimization

Decision variable: depot charging power `P[d,t]`. Objective: minimize hourly electricity cost. Constraints: availability, installed charger rating, daily battery-energy requirement and the network-safe limits established in Step 3.

In [ ]:
schedule, optimization_summary = optimize_cost(fleet, network_limits, prices)
display(optimization_summary)


### What this cell does — AC and contingency validation

Runs the optimized schedule through the same all-closed Pandapower model. N-1 is evaluated at the peak optimized charging hour. The standardized metrics allow direct comparison with uncontrolled charging.

In [ ]:
results = run_timeseries(network_factory, base_p, base_q, schedule, bus_map)
peak_time = results.loc[results.total_ev_charging_MW.idxmax(), "time"]
peak_net = build_snapshot_network(network_factory, base_p, base_q, schedule, bus_map, peak_time)
n1_summary, n1_violations = run_n1(peak_net, switch_configs=SWITCH_CONFIGS_ALL_CLOSED)
metrics = scenario_metrics("cost_optimized", schedule, prices, results, n1_summary)
print("Peak optimized charging hour:", peak_time)
display(metrics)


### What this cell does — Compare with uncontrolled charging

Reports cost and peak-demand changes rather than judging the optimization objective alone. Demand-charge savings cannot be monetized unless a demand tariff is supplied, but the peak-MW change can be reported.

In [ ]:
u_path = config.RESULTS / "step3_uncontrolled" / "scenario_metrics.csv"
if u_path.exists():
    uncontrolled = pd.read_csv(u_path)
    comparison = pd.concat([uncontrolled, metrics], ignore_index=True, sort=False)
    display(comparison)
    print("Peak EV change [MW]:", float(metrics.peak_ev_charging_MW.iloc[0] - uncontrolled.peak_ev_charging_MW.iloc[0]))
    print("Cost change [EUR]:", float(metrics.electricity_cost_EUR.iloc[0] - uncontrolled.electricity_cost_EUR.iloc[0]))


### What this cell does — Export Step 4

Saves the optimal schedule and its independent AC/N-1 validation.

In [ ]:
out = config.RESULTS / "step4_cost_optimization"
out.mkdir(parents=True, exist_ok=True)
schedule.to_csv(out / "cost_optimized_schedule.csv", index=False)
optimization_summary.to_csv(out / "optimization_summary.csv", index=False)
results.to_csv(out / "network_results.csv", index=False)
metrics.to_csv(out / "scenario_metrics.csv", index=False)
n1_summary.to_csv(out / "n1_summary.csv", index=False)
n1_violations.to_csv(out / "n1_violations.csv", index=False)
save_schedule_plot(schedule, out / "cost_schedule.png", "Step 4 — Cost-optimal charging")
save_network_plot(results, out / "network_loading.png", "Step 4 — Network loading")
print("Saved to", out)
